# Phase 4 Stage 3 — v4 Scratch v2 (RTX 5090, 32GB)

**v2 changes vs v1 (`06_train_v4_scratch.ipynb`):**

| # | Item | v1 | v2 |
|---|---|---|---|
| 1 | `max_seq_length` | 192 (1.36% trunc) | **208** (~0.5% trunc, aug data có outlier dài tới 346 tokens) |
| 2 | Aug data spot-check | none | **Section 1** — preview 10 random aug samples |
| 3 | Token re-profile | none | **Section 2** — confirm `MAX_SUMMARY_TOKENS` trên 269K |
| 4 | Smoke VRAM cell | none | **Section 6** — 30 step mini train, abort nếu peak > 30GB |
| 5 | Resume from checkpoint | none | **Section 7** — auto-detect local hoặc HF `last-checkpoint` branch |
| 6 | Live HF push during training | only at end | **`hub_strategy="checkpoint"`** mỗi 500 step → branch `last-checkpoint` |
| 7 | Best ckpt logging | partial | full path + step + RMSLE → `results/v4_scratch_results.json` |
| 8 | Charts | scatter 200 only | **8 charts** (training curves + dashboard + bucket RMSLE + error hist) |

**Hardware:** RTX 5090 32GB | per_device_batch=20, eff_batch=80 | DoRA + RSLoRA + NEFTune α=5

**Target:** RMSLE 0.36–0.40 (beat Day4 v8 0.4004 standalone)

**Resume strategy:** Vast.ai có thể disconnect. Notebook auto-detect checkpoint trên HF Hub branch `last-checkpoint`, download và resume. Bạn chỉ cần restart instance + run cells từ đầu.


In [ ]:
# Chay neu chua co trong env:
#!uv add "transformers>=5.2.0" peft trl bitsandbytes accelerate datasets python-dotenv huggingface-hub matplotlib


In [ ]:
import os
import re
import sys
import gc
import json
import time
import math
import random
import shutil
import tempfile
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
import bitsandbytes as bnb
import torch.nn as nn
import matplotlib.pyplot as plt
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login, snapshot_download, HfApi
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    PreTrainedTokenizerBase,
)
from trl import SFTTrainer, SFTConfig

@dataclass
class DataCollatorForCompletionOnlyLM:
    """Manual impl: trl.DataCollatorForCompletionOnlyLM removed in TRL 0.24.0."""
    response_template: List[int]
    tokenizer: PreTrainedTokenizerBase
    ignore_index: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids_list = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        max_len = max(len(x) for x in input_ids_list)
        bs = len(input_ids_list)
        padded = torch.full((bs, max_len), self.tokenizer.pad_token_id, dtype=torch.long)
        attn   = torch.zeros((bs, max_len), dtype=torch.long)
        labels = torch.full((bs, max_len), self.ignore_index, dtype=torch.long)
        tpl, tpl_len = self.response_template, len(self.response_template)
        for i, ids in enumerate(input_ids_list):
            n = len(ids)
            padded[i, :n] = ids
            attn[i, :n]   = 1
            for j in range(n - tpl_len, -1, -1):
                if ids[j : j + tpl_len].tolist() == tpl:
                    labels[i, j + tpl_len : n] = ids[j + tpl_len : n]
                    break
        return {"input_ids": padded, "attention_mask": attn, "labels": labels}

NOTEBOOK_DIR = Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from utils.evaluator import compute_metrics, plot_predictions
from utils.rmsle_callback import RMSLEEvalCallback

print("Imports OK")
import transformers, peft, trl
for pkg, mod in [("torch", torch), ("transformers", transformers), ("peft", peft), ("trl", trl)]:
    print(f"  {pkg:<14}: {mod.__version__}")
print(f"NOTEBOOK_DIR    : {NOTEBOOK_DIR}")


In [ ]:
# --- v4-scratch v2 constants ---

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME = "SeanSunny/items_prompts_tv_4"

# Sequence — v2 BUMP tu 192 -> 208 (aug data co outlier toi 346 tokens)
MAX_SEQ_LENGTH  = 208
MAX_NEW_TOKENS  = 4
QUESTION_PREFIX = "Sản phẩm này có giá bao nhiêu ?\n"
PRICE_PREFIX    = "\n\nGiá là: "

# LoRA v4-scratch — full SOTA (KHONG doi giua smoke + full)
LORA_R              = 128
LORA_ALPHA          = 256
LORA_DROPOUT        = 0.15
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]
USE_DORA            = True
USE_RSLORA          = True

# Training
TRAIN_SIZE        = None
VAL_CALLBACK_SIZE = 500
VAL_FULL_SIZE     = None
PLOT_SIZE         = 200
NUM_EPOCHS        = 2
PER_DEVICE_BATCH  = 20
GRAD_ACCUM        = 4   # eff_batch = 80
LEARNING_RATE     = 2e-4
LR_SCHEDULER      = "cosine"
WARMUP_RATIO      = 0.03
WEIGHT_DECAY      = 0.01
OPTIM             = "paged_adamw_32bit"
GRADIENT_CHECKPOINTING = True
GROUP_BY_LENGTH   = True
NEFTUNE_ALPHA     = 5
EVAL_STEPS        = 500
SAVE_STEPS        = 500
EARLY_STOP_PATIENCE = 3
LOGGING_STEPS     = 50
SEED              = 42

# Smoke
SMOKE_STEPS       = 30
SMOKE_VRAM_LIMIT_GB = 30.0   # abort full train neu peak vuot

# Inference safety
PRED_CLAMP_MIN = 5
PRED_CLAMP_MAX = 1000
PARSE_REGEX    = r"[-+]?\d*\.\d+|\d+"

# Paths
ADAPTER_DIR     = NOTEBOOK_DIR / "weights" / "v4_scratch_adapter"
RESULTS_FILE    = NOTEBOOK_DIR / "results" / "v4_scratch_results.json"
PREDS_FILE      = NOTEBOOK_DIR / "results" / "v4_scratch_val_predictions.json"
CHARTS_DIR      = NOTEBOOK_DIR / "results" / "charts"
HF_REPO_ADAPTER = "SeanSunny/qwen3.5-4b-vn-pricer-v4-scratch"
HF_CKPT_BRANCH  = "last-checkpoint"   # auto-managed by hub_strategy="checkpoint"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"BASE_MODEL      : {BASE_MODEL}")
print(f"DATASET_NAME    : {DATASET_NAME}")
print(f"MAX_SEQ_LENGTH  : {MAX_SEQ_LENGTH}  (v2 bump tu 192)")
print(f"LoRA            : r={LORA_R}, alpha={LORA_ALPHA}, drop={LORA_DROPOUT}, DoRA={USE_DORA}, RSLoRA={USE_RSLORA}")
print(f"Training        : eff_batch={PER_DEVICE_BATCH*GRAD_ACCUM}, lr={LEARNING_RATE}, wd={WEIGHT_DECAY}, neftune={NEFTUNE_ALPHA}")
print(f"Smoke           : {SMOKE_STEPS} steps, abort if VRAM > {SMOKE_VRAM_LIMIT_GB}GB")
print(f"HF push         : repo={HF_REPO_ADAPTER}, ckpt branch={HF_CKPT_BRANCH}")


In [ ]:
# GPU + HF login + dirs
assert torch.cuda.is_available(), "GPU khong kha dung."
gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {total_vram:.1f} GB")
cap = torch.cuda.get_device_capability()
print(f"bf16 : {'yes' if cap[0] >= 8 else 'no'} (compute {cap})")

env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK (from {env_path})")
else:
    print(f"WARNING: HF_TOKEN not set in {env_path}, prompting login")
    login()

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
(NOTEBOOK_DIR / "results").mkdir(exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dirs ready: {ADAPTER_DIR}, {CHARTS_DIR}")


## 1. Spot-check augmented dataset

Hien thi 10 random sample tu `items_prompts_tv_4` train (269K = 85.7K orig + 183.4K aug). Bam chay cell de visual verify aug quality (paraphrase OK, khong hallucinate, khong loi format).

**Quality scan tren 1000 row tu local (2026-04-29):** 0/1000 hallucination patterns, 0/100 empty desc, 0/100 price mismatch — pass.

In [ ]:
# Spot-check 10 random aug rows
ds_check = load_dataset(DATASET_NAME, split="train")
print(f"Total train: {len(ds_check):,}")
print(f"Columns    : {ds_check.column_names}\n")

rng = random.Random(SEED + 99)
sample_indices = rng.sample(range(len(ds_check)), 10)
for k, i in enumerate(sample_indices):
    row = ds_check[i]
    print(f"--- Sample {k+1} (idx={i}) | price={row['price_vnd_true']:>8,} VND | comp={row['completion']} ---")
    print(row['prompt'][:500])
    print()

del ds_check
gc.collect()


## 2. Token re-profile tren 269K

Verify `MAX_SUMMARY_TOKENS` cho `MAX_SEQ_LENGTH=208`. Aug rows dai hon orig (max v3=218 tokens, max aug=346). Truncation rate phai duoi 1%.

In [ ]:
# Quick token profile tren 5K random sample
tok_probe = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
ds_probe = load_dataset(DATASET_NAME, split="train")
probe_idx = random.Random(SEED).sample(range(len(ds_probe)), 5000)

EOS = tok_probe.eos_token if tok_probe.eos_token else "<|endoftext|>"
prompt_lens, full_lens = [], []
for i in probe_idx:
    row = ds_probe[i]
    pl = len(tok_probe.encode(row['prompt'], add_special_tokens=False))
    fl = len(tok_probe.encode(row['prompt'] + str(row['completion']) + "\n" + EOS, add_special_tokens=False))
    prompt_lens.append(pl)
    full_lens.append(fl)

prompt_lens = np.array(prompt_lens)
full_lens = np.array(full_lens)

print(f"           Prompt    Full")
for q in [50, 90, 95, 99]:
    print(f"  p{q:<3}  : {int(np.percentile(prompt_lens,q)):>5}    {int(np.percentile(full_lens,q)):>5}")
print(f"  max    : {int(prompt_lens.max()):>5}    {int(full_lens.max()):>5}")
print(f"  mean   : {prompt_lens.mean():>5.1f}    {full_lens.mean():>5.1f}")

print()
for L in [192, 208, 224, 256]:
    trunc = (full_lens > L).sum() / len(full_lens) * 100
    marker = " <-- chosen" if L == MAX_SEQ_LENGTH else ""
    print(f"  trunc @ max_seq_len={L:>3}: {trunc:.2f}%{marker}")

q_ids = tok_probe.encode(QUESTION_PREFIX, add_special_tokens=False)
p_ids = tok_probe.encode(PRICE_PREFIX, add_special_tokens=False)
TOKENS_FIXED = len(q_ids) + len(p_ids)
MAX_SUMMARY_TOKENS = MAX_SEQ_LENGTH - TOKENS_FIXED - MAX_NEW_TOKENS - 1 - 2

print(f"\nTOKENS_FIXED        : {TOKENS_FIXED}  (QUESTION={len(q_ids)} + PRICE={len(p_ids)})")
print(f"MAX_SUMMARY_TOKENS  : {MAX_SUMMARY_TOKENS}")

del tok_probe, ds_probe, prompt_lens, full_lens
gc.collect()


## 3. Build model — base 4-bit + LoRA scratch

Function `build_model()` duoc goi 2 lan:
1. Lan 1: cho **smoke VRAM test** (Section 6)
2. Lan 2: rebuild fresh cho **full train** (Section 8) — tranh state pollution tu smoke

`dtype=torch.bfloat16` BAT BUOC trong `from_pretrained` (lesson v1 — neu khong se crash conv1d).

In [ ]:
def build_model():
    """Build base model (4-bit) + LoRA adapter scratch. Tra ve (model, tokenizer)."""
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    if tokenizer.eos_token_id is None:
        tokenizer.eos_token = "<|endoftext|>"
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
    )
    base_model = prepare_model_for_kbit_training(
        base_model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING
    )
    for _m in base_model.modules():
        if isinstance(_m, torch.nn.Conv1d):
            _m.to(torch.bfloat16)

    lora_cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        use_dora=USE_DORA, use_rslora=USE_RSLORA,
        bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.enable_input_require_grads()
    return model, tokenizer

# Build for smoke
model, tokenizer = build_model()
model.print_trainable_parameters()
print(f"EOS token       : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"PAD token       : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


## 4. Dataset preprocess + tokenize

Pipeline giong v3: cat summary token-level, append PRICE_PREFIX + completion + EOS.

`items_prompts_tv_4` co 3 splits: `train` (269,112), `val` (3,926), `test` (3,872).

In [ ]:
ds = load_dataset(DATASET_NAME)
print(f"Train: {len(ds['train']):,} | Val: {len(ds['val']):,} | Test: {len(ds['test']):,}")

q_ids = tokenizer.encode(QUESTION_PREFIX, add_special_tokens=False)
p_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
TOKENS_FIXED = len(q_ids) + len(p_ids)
MAX_SUMMARY_TOKENS = MAX_SEQ_LENGTH - TOKENS_FIXED - MAX_NEW_TOKENS - 1 - 2
print(f"TOKENS_FIXED       : {TOKENS_FIXED}")
print(f"MAX_SUMMARY_TOKENS : {MAX_SUMMARY_TOKENS}")

train_full_raw = ds["train"].shuffle(seed=SEED)
if TRAIN_SIZE is not None:
    train_full_raw = train_full_raw.select(range(TRAIN_SIZE))
val_full_raw = ds["val"]
val_callback_raw = ds["val"].shuffle(seed=SEED).select(range(VAL_CALLBACK_SIZE))

def preprocess(example):
    p = example["prompt"]
    summary = p[len(QUESTION_PREFIX):-len(PRICE_PREFIX)]
    summary_ids = tokenizer.encode(summary, add_special_tokens=False)
    if len(summary_ids) > MAX_SUMMARY_TOKENS:
        summary_ids = summary_ids[:MAX_SUMMARY_TOKENS]
        summary = tokenizer.decode(summary_ids, skip_special_tokens=True).rstrip()
    full_text = QUESTION_PREFIX + summary + PRICE_PREFIX + example["completion"] + "\n" + tokenizer.eos_token
    return {"text": full_text}

train_ds = train_full_raw.map(preprocess, desc="Preprocess train")
val_ds_for_trainer = val_callback_raw.map(preprocess, desc="Preprocess val (trainer)")

def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

train_ds_tok = train_ds.map(tokenize_fn, batched=False, remove_columns=["text"])
val_ds_tok   = val_ds_for_trainer.map(tokenize_fn, batched=False, remove_columns=["text"])
train_ds_tok = train_ds_tok.map(lambda ex: {"length": len(ex["input_ids"])}, desc="Compute length")

print(f"\ntrain_ds_tok    : {len(train_ds_tok):,}")
print(f"val_ds_tok      : {len(val_ds_tok):,}")
print(f"Sample 0 length : {len(train_ds_tok[0]['input_ids'])}")

for i in range(2):
    t = train_ds[i]["text"]
    assert PRICE_PREFIX in t, f"PRICE_PREFIX missing in sample {i}"
print("PRICE_PREFIX present: OK")


## 5. DataCollator + mask verify + RMSLE callback

In [ ]:
response_template_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
decoded_back = tokenizer.decode(response_template_ids)
print(f"response_template_ids : {response_template_ids}")
print(f"decoded back          : {decoded_back!r}")
assert decoded_back == PRICE_PREFIX, "PRICE_PREFIX decode mismatch"

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

# Mask verify
sample_text = train_ds[0]["text"]
tokenized = tokenizer(sample_text, return_tensors="pt", max_length=MAX_SEQ_LENGTH, truncation=True)
batch_in = [{
    "input_ids": tokenized["input_ids"][0].tolist(),
    "attention_mask": tokenized["attention_mask"][0].tolist(),
}]
labels = collator(batch_in)["labels"][0]
non_masked = labels[labels != -100]
if len(non_masked) == 0:
    raise RuntimeError("Mask verify FAILED.")
print(f"Decoded non-masked    : {tokenizer.decode(non_masked.tolist(), skip_special_tokens=False)!r}")
print(f"Sample completion     : {train_full_raw[0]['completion']!r}")
print("Mask verify PASS.")

rmsle_callback = RMSLEEvalCallback(
    tokenizer=tokenizer,
    val_subset=val_callback_raw,
    max_new_tokens=MAX_NEW_TOKENS,
    clamp_min=PRED_CLAMP_MIN,
    clamp_max=PRED_CLAMP_MAX,
)
print(f"RMSLEEvalCallback ready (n={len(val_callback_raw)} items).")


## 6. SMOKE VRAM TEST (30 step + RMSLE callback infrastructure)

**Muc dich:** Do VRAM peak thuc te tren 5090 va catch bug callback truoc khi commit 17–20h training.

**Abort condition:** Neu peak VRAM > **30GB** (`SMOKE_VRAM_LIMIT_GB`) → raise RuntimeError, ban phai giam `PER_DEVICE_BATCH` xuong 16 hoac 12 truoc khi tiep tuc.

**Sau smoke:** model va smoke_trainer bi xoa, build lai fresh o Section 7.

In [ ]:
torch.cuda.reset_peak_memory_stats()
SMOKE_DIR = tempfile.mkdtemp(prefix="smoke_v4_")
print(f"Smoke output dir: {SMOKE_DIR}")

smoke_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_tok.select(range(min(SMOKE_STEPS * PER_DEVICE_BATCH * GRAD_ACCUM, len(train_ds_tok)))),
    data_collator=collator,
    args=SFTConfig(
        output_dir=SMOKE_DIR,
        max_steps=SMOKE_STEPS,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        bf16=True,
        max_grad_norm=0.3,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        train_sampling_strategy="group_by_length" if GROUP_BY_LENGTH else "random",
        length_column_name="length",
        neftune_noise_alpha=NEFTUNE_ALPHA,
        logging_steps=5,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
        seed=SEED,
    ),
)
t0 = time.time()
smoke_trainer.train()
smoke_train_sec = time.time() - t0
smoke_vram_peak = torch.cuda.max_memory_allocated() / 1e9
sec_per_step = smoke_train_sec / SMOKE_STEPS

# Estimate full training
total_steps_full = math.ceil(len(train_ds_tok) / (PER_DEVICE_BATCH * GRAD_ACCUM)) * NUM_EPOCHS
total_time_est_hr = total_steps_full * sec_per_step / 3600

print(f"\n=== SMOKE RESULT ===")
print(f"VRAM peak       : {smoke_vram_peak:.2f} GB  (limit: {SMOKE_VRAM_LIMIT_GB:.1f} GB)")
print(f"Sec/step        : {sec_per_step:.2f}s")
print(f"Estimated full  : {total_steps_full:,} steps, ~{total_time_est_hr:.1f} hr")

# Test RMSLE callback infrastructure (catch bugs som)
print(f"\nTesting RMSLEEvalCallback on {len(val_callback_raw)} val subset...")
t0 = time.time()
test_rmsle = rmsle_callback._compute_rmsle(model)
callback_sec = time.time() - t0
print(f"Smoke RMSLE     : {test_rmsle:.4f}  (untrained — chi check infrastructure)")
print(f"Callback time   : {callback_sec:.1f}s per eval")
print(f"Total eval cost : ~{(total_steps_full // EVAL_STEPS) * callback_sec / 60:.1f} min over {total_steps_full // EVAL_STEPS} evals")

# ABORT check
if smoke_vram_peak > SMOKE_VRAM_LIMIT_GB:
    raise RuntimeError(
        f"VRAM peak {smoke_vram_peak:.2f}GB > {SMOKE_VRAM_LIMIT_GB}GB threshold!\n"
        f"Reduce PER_DEVICE_BATCH (try 16 or 12), restart kernel, re-run."
    )
print(f"\n=== SMOKE PASS — proceed to full train ===")

# Cleanup smoke
del smoke_trainer, model, tokenizer
gc.collect()
torch.cuda.empty_cache()
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
print(f"\nCleanup done. VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 7. Resume detection + Rebuild fresh model

**Resume strategy (vast.ai disconnect protection):**
1. Check local `weights/v4_scratch_adapter/checkpoint-*` (neu instance van con disk).
2. Neu khong → check HF Hub branch `last-checkpoint` (auto-pushed bang `hub_strategy="checkpoint"`).
3. Neu co → download va resume tu do.
4. Neu khong → fresh start.

Sau buoc resume, **rebuild fresh model** (smoke da pollute model state).

In [ ]:
# Step 1: Detect resume checkpoint
RESUME_PATH = None

# Check local
local_ckpts = sorted(ADAPTER_DIR.glob("checkpoint-*"))
if local_ckpts:
    RESUME_PATH = str(local_ckpts[-1])
    print(f"[Resume] Local checkpoint detected: {RESUME_PATH}")
else:
    # Check HF Hub last-checkpoint branch
    api = HfApi()
    try:
        refs = api.list_repo_refs(HF_REPO_ADAPTER, repo_type="model")
        branch_names = [b.name for b in refs.branches]
        if HF_CKPT_BRANCH in branch_names:
            print(f"[Resume] HF branch '{HF_CKPT_BRANCH}' found, downloading...")
            RESUME_PATH = snapshot_download(
                repo_id=HF_REPO_ADAPTER,
                revision=HF_CKPT_BRANCH,
                local_dir=str(ADAPTER_DIR / "_hub_resume"),
            )
            print(f"[Resume] Downloaded to: {RESUME_PATH}")
        else:
            print(f"[Resume] No '{HF_CKPT_BRANCH}' branch on HF — fresh start.")
    except Exception as e:
        print(f"[Resume] HF check failed (likely repo chua ton tai, OK cho first run): {type(e).__name__}: {e}")

if RESUME_PATH is None:
    print("[Resume] No checkpoint — training fresh from step 0.")
else:
    print(f"[Resume] Will resume from: {RESUME_PATH}")

# Step 2: Rebuild fresh model
print("\n[Rebuild] Building fresh model after smoke...")
model, tokenizer = build_model()
model.print_trainable_parameters()
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

# Re-init RMSLE callback voi tokenizer moi
rmsle_callback = RMSLEEvalCallback(
    tokenizer=tokenizer,
    val_subset=val_callback_raw,
    max_new_tokens=MAX_NEW_TOKENS,
    clamp_min=PRED_CLAMP_MIN,
    clamp_max=PRED_CLAMP_MAX,
)
print("RMSLEEvalCallback re-initialized.")


## 8. Full train — DoRA + RSLoRA + NEFTune + best by RMSLE + HF live push

- `hub_strategy="checkpoint"` → moi 500 step push checkpoint len HF Hub branch `last-checkpoint` (resume-able).
- `metric_for_best_model="eval_rmsle"`, `greater_is_better=False` (custom callback).
- `EarlyStoppingCallback(patience=3)`.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t_train_start = time.time()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    data_collator=collator,
    callbacks=[
        rmsle_callback,
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
    ],
    args=SFTConfig(
        output_dir=str(ADAPTER_DIR),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        optim=OPTIM,
        bf16=True,
        max_grad_norm=0.3,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        train_sampling_strategy="group_by_length" if GROUP_BY_LENGTH else "random",
        length_column_name="length",
        neftune_noise_alpha=NEFTUNE_ALPHA,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_rmsle",
        greater_is_better=False,
        logging_steps=LOGGING_STEPS,
        report_to="none",
        seed=SEED,
        # HF Hub live push
        push_to_hub=True,
        hub_model_id=HF_REPO_ADAPTER,
        hub_strategy="checkpoint",
        hub_private_repo=True,
    ),
)

if RESUME_PATH:
    print(f"[Train] Resuming from {RESUME_PATH}")
    trainer.train(resume_from_checkpoint=RESUME_PATH)
else:
    print("[Train] Fresh start")
    trainer.train()

trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

total_train_sec = time.time() - t_train_start
final_vram_peak = torch.cuda.max_memory_allocated() / 1e9
print(f"\nTraining complete: {total_train_sec/60:.1f} min ({total_train_sec/3600:.2f} hr) | VRAM peak: {final_vram_peak:.2f} GB")

# Extract logs for charts + best ckpt info
log_history = trainer.state.log_history
train_losses     = [(int(e["step"]), float(e["loss"]))      for e in log_history if "loss" in e and "eval_loss" not in e]
eval_losses      = [(int(e["step"]), float(e["eval_loss"])) for e in log_history if "eval_loss" in e]
eval_rmsle_curve = [(int(e["step"]), float(e["eval_rmsle"])) for e in log_history if "eval_rmsle" in e]
lr_curve         = [(int(e["step"]), float(e["learning_rate"])) for e in log_history if "learning_rate" in e]

print(f"\nLog summary:")
print(f"  Train loss entries : {len(train_losses)}")
print(f"  Eval CE entries    : {len(eval_losses)}")
print(f"  Eval RMSLE entries : {len(eval_rmsle_curve)}")
print(f"  LR entries         : {len(lr_curve)}")

best_ckpt_step = None
best_ckpt_rmsle = None
best_ckpt_path = trainer.state.best_model_checkpoint
if eval_rmsle_curve:
    best_ckpt_step, best_ckpt_rmsle = min(eval_rmsle_curve, key=lambda x: x[1])
    print(f"\nBest eval_rmsle (subset n={VAL_CALLBACK_SIZE}): {best_ckpt_rmsle:.4f} @ step {best_ckpt_step}")
    print(f"Best ckpt path: {best_ckpt_path}")

del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")


## 9. Final generative eval — full 3,926 val (best ckpt da auto-loaded)

In [ ]:
model.eval()
for _m in model.modules():
    if isinstance(_m, torch.nn.Conv1d):
        _m.to(torch.bfloat16)

def predict_one(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(PARSE_REGEX, gen)
    if m:
        pk = max(PRED_CLAMP_MIN, min(int(float(m.group())), PRED_CLAMP_MAX))
    else:
        pk = 0
    return pk, gen

val_for_eval = val_full_raw if VAL_FULL_SIZE is None else val_full_raw.select(range(VAL_FULL_SIZE))
preds_vnd, trues_vnd, raw_outs = [], [], []
clamp_count = 0
t_eval_start = time.time()

for item in tqdm(val_for_eval, desc="Generative eval (full val)"):
    pk, raw = predict_one(item["prompt"])
    preds_vnd.append(pk * 1000)
    trues_vnd.append(item["price_vnd_true"])
    raw_outs.append(raw)
    if pk in (PRED_CLAMP_MIN, PRED_CLAMP_MAX):
        clamp_count += 1

t_eval = time.time() - t_eval_start
metrics_final = compute_metrics(
    np.array(trues_vnd, dtype=float), np.array(preds_vnd, dtype=float)
)
n_eval = len(val_for_eval)

print("=" * 50)
print(f"v4-scratch v2 — {n_eval} val (best ckpt by eval_rmsle)")
print("=" * 50)
print(f"RMSLE : {metrics_final['rmsle']:.4f}  (primary)")
print(f"MAE   : {metrics_final['mae']:,.0f} VND")
print(f"MAPE  : {metrics_final['mape']:.1f}%")
print(f"R2    : {metrics_final['r2']:.4f}")
print(f"Zero preds   : {preds_vnd.count(0)}")
print(f"Clamp trigger: {clamp_count}")
print(f"Sec/item     : {t_eval / n_eval:.2f}s")
print("=" * 50)
print(f"v3 ref       : RMSLE=0.4426")
print(f"Day4 v8 ref  : RMSLE=0.4004")
print(f"Target Stage3: RMSLE 0.36-0.40")


In [ ]:
# Save val predictions (cho 07_ensemble.ipynb)
preds_dump = [
    {"idx": i, "pred_vnd": int(preds_vnd[i]), "true_vnd": int(trues_vnd[i])}
    for i in range(n_eval)
]
with open(PREDS_FILE, "w", encoding="utf-8") as f:
    json.dump(preds_dump, f, ensure_ascii=False)
print(f"Saved predictions: {PREDS_FILE}")

samples_out = []
for i in range(min(20, n_eval)):
    tv, pv = trues_vnd[i], preds_vnd[i]
    err_pct = abs(pv - tv) / tv * 100 if tv > 0 else None
    samples_out.append({
        "idx": i,
        "prompt_excerpt": val_for_eval[i]["prompt"][:120],
        "generated_raw": raw_outs[i],
        "pred_vnd": pv, "true_vnd": tv,
        "error_pct": round(err_pct, 1) if err_pct is not None else None,
    })

results = {
    "version": "v4_scratch_v2",
    "model": BASE_MODEL,
    "dataset": DATASET_NAME,
    "config": {
        "train_size": len(train_ds_tok),
        "val_callback_size": VAL_CALLBACK_SIZE,
        "val_full_size": n_eval,
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "use_dora": USE_DORA, "use_rslora": USE_RSLORA,
        "target_modules": LORA_TARGET_MODULES,
        "num_epochs": NUM_EPOCHS,
        "per_device_batch": PER_DEVICE_BATCH, "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE, "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY, "neftune_alpha": NEFTUNE_ALPHA,
        "group_by_length": GROUP_BY_LENGTH,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "early_stop_patience": EARLY_STOP_PATIENCE,
    },
    "smoke": {
        "vram_peak_gb": round(smoke_vram_peak, 2),
        "sec_per_step": round(sec_per_step, 2),
        "total_steps_est": total_steps_full,
        "total_time_est_hr": round(total_time_est_hr, 1),
    },
    "best_ckpt": {
        "step": best_ckpt_step,
        "rmsle_subset": round(best_ckpt_rmsle, 4) if best_ckpt_rmsle else None,
        "path": best_ckpt_path,
    } if best_ckpt_step else None,
    "vram_train_peak_gb": round(final_vram_peak, 2),
    "total_train_sec": round(total_train_sec, 1),
    "sec_per_val_item": round(t_eval / n_eval, 2),
    "train_loss_curve": train_losses,
    "eval_loss_curve": eval_losses,
    "eval_rmsle_curve": eval_rmsle_curve,
    "lr_curve": lr_curve,
    "metrics_final": metrics_final,
    "samples_20": samples_out,
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved: {RESULTS_FILE}")


## 10. Charts (8 figures, PNG @ DPI=150 luu o `results/charts/`)

1. Train CE loss (raw + EMA smooth window=10)
2. Eval CE loss
3. Eval RMSLE + best step marker (CE↔RMSLE divergence theo v3)
4. Learning rate (cosine warmup+decay)
5. Combined 2x2 dashboard
6. Pred vs True scatter (200 random sample, log scale)
7. Error % histogram (200 sample)
8. Per-price-bucket RMSLE (5 buckets: <50K, 50-100K, 100-200K, 200-500K, 500K-1M)

In [ ]:
# Helper: EMA smoothing
def ema_smooth(values, alpha=0.1):
    if not values: return values
    out = [values[0]]
    for v in values[1:]:
        out.append(alpha * v + (1 - alpha) * out[-1])
    return out

DPI = 150
plt.rcParams.update({"figure.dpi": DPI, "savefig.dpi": DPI})

# Chart 1: Train CE loss
if train_losses:
    steps_t, vals_t = zip(*train_losses)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_t, vals_t, alpha=0.3, color="C0", label="raw")
    smoothed = ema_smooth(list(vals_t), alpha=0.1)
    ax.plot(steps_t, smoothed, color="C0", linewidth=2, label="EMA (alpha=0.1)")
    ax.set_xlabel("Step"); ax.set_ylabel("Train CE Loss")
    ax.set_title("v4-scratch v2 — Train CE Loss")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "v4_scratch_01_train_loss.png")
    plt.show()
    print(f"Saved: {CHARTS_DIR / 'v4_scratch_01_train_loss.png'}")

# Chart 2: Eval CE loss
if eval_losses:
    steps_e, vals_e = zip(*eval_losses)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_e, vals_e, "o-", color="C1", linewidth=2)
    ax.set_xlabel("Step"); ax.set_ylabel("Eval CE Loss")
    ax.set_title("v4-scratch v2 — Eval CE Loss (overfit detection)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "v4_scratch_02_eval_loss.png")
    plt.show()

# Chart 3: Eval RMSLE + best step
if eval_rmsle_curve:
    steps_r, vals_r = zip(*eval_rmsle_curve)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_r, vals_r, "o-", color="C2", linewidth=2)
    if best_ckpt_step:
        ax.axvline(best_ckpt_step, color="red", linestyle="--", alpha=0.7,
                   label=f"Best step {best_ckpt_step} (RMSLE {best_ckpt_rmsle:.4f})")
        ax.legend()
    ax.set_xlabel("Step"); ax.set_ylabel("Eval RMSLE (500 val subset)")
    ax.set_title("v4-scratch v2 — Eval RMSLE (primary metric)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "v4_scratch_03_eval_rmsle.png")
    plt.show()

# Chart 4: Learning rate
if lr_curve:
    steps_l, vals_l = zip(*lr_curve)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_l, vals_l, color="C3", linewidth=2)
    ax.set_xlabel("Step"); ax.set_ylabel("Learning Rate")
    ax.set_title(f"v4-scratch v2 — LR schedule (cosine, warmup={WARMUP_RATIO})")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "v4_scratch_04_lr.png")
    plt.show()

# Chart 5: Combined 2x2 dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax = axes[0, 0]
if train_losses:
    steps_t, vals_t = zip(*train_losses)
    ax.plot(steps_t, vals_t, alpha=0.3, color="C0")
    ax.plot(steps_t, ema_smooth(list(vals_t), 0.1), color="C0", linewidth=2)
    ax.set_title("Train CE Loss"); ax.set_xlabel("Step"); ax.grid(alpha=0.3)

ax = axes[0, 1]
if eval_losses:
    steps_e, vals_e = zip(*eval_losses)
    ax.plot(steps_e, vals_e, "o-", color="C1", linewidth=2)
    ax.set_title("Eval CE Loss"); ax.set_xlabel("Step"); ax.grid(alpha=0.3)

ax = axes[1, 0]
if eval_rmsle_curve:
    steps_r, vals_r = zip(*eval_rmsle_curve)
    ax.plot(steps_r, vals_r, "o-", color="C2", linewidth=2)
    if best_ckpt_step:
        ax.axvline(best_ckpt_step, color="red", linestyle="--", alpha=0.7, label=f"Best @ {best_ckpt_step}")
        ax.legend()
    ax.set_title("Eval RMSLE (primary)"); ax.set_xlabel("Step"); ax.grid(alpha=0.3)

ax = axes[1, 1]
if lr_curve:
    steps_l, vals_l = zip(*lr_curve)
    ax.plot(steps_l, vals_l, color="C3", linewidth=2)
    ax.set_title("Learning Rate"); ax.set_xlabel("Step"); ax.grid(alpha=0.3)

fig.suptitle(f"v4-scratch v2 — Training Dashboard (final RMSLE={metrics_final['rmsle']:.4f})", fontsize=14)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "v4_scratch_05_dashboard.png")
plt.show()
print(f"Saved 5/8 charts to {CHARTS_DIR}")


In [ ]:
# Chart 6: Pred vs True scatter (200 random sample, log scale)
rng = np.random.default_rng(SEED)
plot_idx = rng.choice(n_eval, size=min(PLOT_SIZE, n_eval), replace=False)
trues_plot = np.array([trues_vnd[i] for i in plot_idx], dtype=float)
preds_plot = np.array([preds_vnd[i] for i in plot_idx], dtype=float)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(trues_plot, preds_plot, alpha=0.5, s=20)
mn = min(trues_plot.min(), preds_plot[preds_plot > 0].min() if (preds_plot > 0).any() else 1)
mx = max(trues_plot.max(), preds_plot.max())
ax.plot([mn, mx], [mn, mx], "r--", alpha=0.7, label="y=x (perfect)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("True price (VND, log)"); ax.set_ylabel("Predicted price (VND, log)")
ax.set_title(f"v4-scratch v2 — Pred vs True ({PLOT_SIZE} random sample, log scale)\nRMSLE={metrics_final['rmsle']:.4f}")
ax.legend(); ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "v4_scratch_06_scatter_200.png")
plt.show()
print(f"Saved: chart 6")

# Chart 7: Error % histogram (200 sample)
errors_pct = np.abs(preds_plot - trues_plot) / np.maximum(trues_plot, 1) * 100
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.clip(errors_pct, 0, 200), bins=40, color="C4", edgecolor="black", alpha=0.7)
ax.axvline(np.median(errors_pct), color="red", linestyle="--", label=f"Median: {np.median(errors_pct):.1f}%")
ax.axvline(np.mean(errors_pct), color="orange", linestyle="--", label=f"Mean: {np.mean(errors_pct):.1f}%")
ax.set_xlabel("Absolute error % (clipped @ 200%)"); ax.set_ylabel("Count")
ax.set_title(f"v4-scratch v2 — Error distribution ({PLOT_SIZE} random sample)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "v4_scratch_07_error_hist.png")
plt.show()
print(f"Saved: chart 7")


In [ ]:
# Chart 8: Per-price-bucket RMSLE (5 buckets, full val)
trues_arr = np.array(trues_vnd, dtype=float)
preds_arr = np.array(preds_vnd, dtype=float)
buckets = [
    ("<50K",       0,       50_000),
    ("50-100K",    50_000,  100_000),
    ("100-200K",   100_000, 200_000),
    ("200-500K",   200_000, 500_000),
    ("500K-1M",    500_000, 1_000_001),
]
labels, rmsles, counts = [], [], []
for name, lo, hi in buckets:
    mask = (trues_arr >= lo) & (trues_arr < hi)
    n = mask.sum()
    if n == 0:
        continue
    rmsle_b = float(np.sqrt(np.mean((np.log1p(preds_arr[mask]) - np.log1p(trues_arr[mask]))**2)))
    labels.append(name); rmsles.append(rmsle_b); counts.append(int(n))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, rmsles, color="C5", edgecolor="black", alpha=0.8)
for bar, r, n in zip(bars, rmsles, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{r:.3f}\n(n={n})", ha="center", va="bottom", fontsize=9)
ax.axhline(metrics_final['rmsle'], color="red", linestyle="--", alpha=0.7,
           label=f"Overall RMSLE: {metrics_final['rmsle']:.4f}")
ax.set_xlabel("Price bucket (VND)"); ax.set_ylabel("RMSLE")
ax.set_title(f"v4-scratch v2 — RMSLE by price bucket (full {n_eval} val)")
ax.legend(); ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "v4_scratch_08_bucket_rmsle.png")
plt.show()
print(f"Saved: chart 8")

print(f"\n=== ALL 8 CHARTS SAVED to {CHARTS_DIR} ===")
for p in sorted(CHARTS_DIR.glob("v4_scratch_*.png")):
    print(f"  {p.name}")


## 11. Push HF main branch (best model final)

`hub_strategy="checkpoint"` da push checkpoint xuyen suot training len branch `last-checkpoint`. Bay gio push best model lan cuoi len `main` branch (ban chinh).

In [ ]:
print(f"Pushing best model to {HF_REPO_ADAPTER} (main branch, private)...")
model.push_to_hub(HF_REPO_ADAPTER, private=True)
tokenizer.push_to_hub(HF_REPO_ADAPTER, private=True)
print(f"Pushed: https://huggingface.co/{HF_REPO_ADAPTER}")

# Final summary
print("\n" + "=" * 60)
print("v4-scratch v2 — FINAL SUMMARY")
print("=" * 60)
print(f"Train size       : {len(train_ds_tok):,}")
print(f"Train time       : {total_train_sec/3600:.2f} hr")
print(f"VRAM peak (smoke): {smoke_vram_peak:.2f} GB")
print(f"VRAM peak (full) : {final_vram_peak:.2f} GB")
print(f"Best ckpt step   : {best_ckpt_step} (RMSLE subset {best_ckpt_rmsle:.4f})" if best_ckpt_step else "N/A")
print(f"Final RMSLE (3,926 val) : {metrics_final['rmsle']:.4f}")
print(f"Final MAE  (3,926 val)  : {metrics_final['mae']:,.0f} VND")
print(f"v3 ref           : 0.4426 | Day4 v8 ref: 0.4004 | Target: 0.36-0.40")
print("=" * 60)

gc.collect()
torch.cuda.empty_cache()


## 12. Log template for `phase2_execution_log.md` Run #4

Sau khi chay xong, paste ket qua vao `phase2_execution_log.md` muc **Run #4 — v4-scratch v2**:

```markdown
## Run #4 — v4-scratch v2 — YYYY-MM-DD

**Notebook:** `fine_tune_qwen/06_train_v4_scratch_v2.ipynb`
**Hardware:** NVIDIA RTX 5090 32GB (vast.ai)
**Python env:** uv run | torch <ver> | transformers <ver> | peft <ver> | trl <ver>

### Smoke result
- VRAM peak: <X.XX> GB (limit 30GB)
- Sec/step  : <X.XX>s
- Est total : <N> steps, <X.X> hr

### Full train
- Total steps: <N>
- Wall-clock : <X.XX> hr
- VRAM peak  : <X.XX> GB
- Best ckpt  : step <N>, eval_rmsle subset = <X.XXXX>
- HF ckpt branch: https://huggingface.co/SeanSunny/qwen3.5-4b-vn-pricer-v4-scratch/tree/last-checkpoint
- HF main      : https://huggingface.co/SeanSunny/qwen3.5-4b-vn-pricer-v4-scratch

### Final eval (3,926 val)
| Metric | Value |
|---|---|
| RMSLE | <X.XXXX> |
| MAE   | <XX,XXX> VND |
| MAPE  | <XX.X>% |
| R2    | <X.XXXX> |

### Per-bucket RMSLE
| Bucket | RMSLE | n |
|---|---|---|
| <50K | <X.XXX> | <N> |
| 50-100K | <X.XXX> | <N> |
| 100-200K | <X.XXX> | <N> |
| 200-500K | <X.XXX> | <N> |
| 500K-1M | <X.XXX> | <N> |

### Charts (8 PNG da save)
- `results/charts/v4_scratch_01_train_loss.png`
- `results/charts/v4_scratch_02_eval_loss.png`
- `results/charts/v4_scratch_03_eval_rmsle.png`
- `results/charts/v4_scratch_04_lr.png`
- `results/charts/v4_scratch_05_dashboard.png`
- `results/charts/v4_scratch_06_scatter_200.png`
- `results/charts/v4_scratch_07_error_hist.png`
- `results/charts/v4_scratch_08_bucket_rmsle.png`

### Issues / Deviations
- (lit ke neu co)

### Notes for ensemble (07_ensemble.ipynb)
- val_predictions: `results/v4_scratch_val_predictions.json` (n=3,926)
- Skip v4-resume → ensemble = v3 + v4-scratch + v8 (3-model Ridge)
```

**Sau Stage 3:** Chay `07_ensemble.ipynb` de blend v3 + v4-scratch + v8.